# 09 — Tier-Sweep Aggregation & Paper Tables

Run this **after** `experiments/run_tier_sweep.py` has produced
`experiments/results/tier_comparison.json`. It renders:

1. A headline accuracy-by-tier table (Cohen's kappa, time MAPE) — for **section 5.1**.
2. A full five-axis-by-tier table (mean +/- std across runs) — for an **appendix**.
3. A ready-to-paste LaTeX version of both (booktabs).

No API calls are made here; it only reads the aggregated JSON.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'experiments' / 'results'

summary = json.loads((RESULTS / 'tier_comparison.json').read_text(encoding='utf-8'))
TIERS = list(summary['tiers'].keys())
print('Tiers:', TIERS)
print('Seeds:', summary['seeds'])
print('Matcher tier (fixed):', summary['matcher_tier'])

In [ ]:
def cell(tier, metric, pct=False, nd=2):
    a = summary['tiers'][tier].get(metric, {})
    m, s = a.get('mean'), a.get('std')
    if m is None:
        return 'n/a'
    suffix = '%' if pct else ''
    if s in (None, 0.0):
        return f'{m:.{nd}f}{suffix}'
    return f'{m:.{nd}f} \u00b1 {s:.{nd}f}{suffix}'

# ---- Headline accuracy table (section 5.1) -------------------------------
head_rows = []
for t in TIERS:
    head_rows.append({
        'Tier': t,
        "Grade agreement": cell(t, 'acc.grade_agreement', nd=3),
        "Cohen's kappa": cell(t, 'acc.grade_cohen_kappa', nd=3),
        'Time MAE (min)': cell(t, 'acc.time_MAE_min', nd=1),
        'Time MAPE (%)': cell(t, 'acc.time_MAPE_pct', pct=True, nd=1),
    })
head = pd.DataFrame(head_rows)
print('HEADLINE ACCURACY BY TIER (mean +/- std across runs)')
print(head.to_string(index=False))

In [ ]:
# ---- Full five-axis table (appendix) -------------------------------------
METRIC_LABELS = [
    ('acc.grade_cohen_kappa', "Accuracy: grade kappa", False, 3),
    ('acc.time_MAPE_pct', 'Accuracy: time MAPE', True, 1),
    ('rel.mean_cv_minutes', 'Reliability: mean CV', False, 3),
    ('rel.pct_nodes_low_cv', 'Reliability: low-CV nodes', True, 1),
    ('eff.annual_saving_usd', 'Efficiency: annual saving (USD)', False, 0),
    ('eff.pipeline_llm_cost_usd', 'Efficiency: pipeline cost (USD)', False, 4),
    ('tra.grade_rationale_coverage_pct', 'Transparency: rationale coverage', True, 1),
    ('tra.pct_estimates_grounded', 'Transparency: grounded share', True, 1),
    ('rob.clarifying_rate_pct', 'Robustness: clarifying rate', True, 1),
]
full_rows = []
for metric, label, pct, nd in METRIC_LABELS:
    row = {'Metric': label}
    for t in TIERS:
        row[t] = cell(t, metric, pct=pct, nd=nd)
    full_rows.append(row)
full = pd.DataFrame(full_rows)
print('FULL FIVE-AXIS BY TIER')
print(full.to_string(index=False))

full.to_csv(RESULTS / 'paper_full_table.csv', index=False, encoding='utf-8-sig')
head.to_csv(RESULTS / 'paper_headline_table.csv', index=False, encoding='utf-8-sig')
print('\nSaved paper_headline_table.csv and paper_full_table.csv')

In [ ]:
# ---- LaTeX (booktabs) for direct pasting ---------------------------------
def to_latex(df, caption, label):
    cols = list(df.columns)
    align = 'l' + 'c' * (len(cols) - 1)
    lines = [r'\begin{table}[H]\centering', r'\small',
             f'\\caption{{{caption}}}', f'\\label{{{label}}}',
             f'\\begin{{tabular}}{{@{{}}{align}@{{}}}}', r'\toprule',
             ' & '.join(cols) + r' \\', r'\midrule']
    for _, r in df.iterrows():
        lines.append(' & '.join(str(r[c]) for c in cols) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

tex_head = to_latex(head,
    'Accuracy of the pipeline against the partial expert reference across model '
    'tiers (mean $\\pm$ standard deviation over five runs; the accuracy-axis '
    'matcher is held at the strong tier throughout).',
    'tab:tier_accuracy')
tex_full = to_latex(full,
    'Five-axis validation metrics by model tier (mean $\\pm$ standard deviation '
    'over five runs).', 'tab:tier_fiveaxis')

(RESULTS / 'paper_headline_table.tex').write_text(tex_head, encoding='utf-8')
(RESULTS / 'paper_full_table.tex').write_text(tex_full, encoding='utf-8')
print(tex_head)
print()
print(tex_full)
print('\nSaved paper_headline_table.tex and paper_full_table.tex')

In [ ]:
# ---- One-line narrative helper for section 5.1 ---------------------------
def m(tier, metric):
    return summary['tiers'][tier].get(metric, {}).get('mean')

if len(TIERS) >= 2:
    k_lo, k_hi = m(TIERS[0], 'acc.grade_cohen_kappa'), m(TIERS[-1], 'acc.grade_cohen_kappa')
    p_lo, p_hi = m(TIERS[0], 'acc.time_MAPE_pct'), m(TIERS[-1], 'acc.time_MAPE_pct')
    if None not in (k_lo, k_hi):
        direction = 'rises' if k_hi > k_lo else ('falls' if k_hi < k_lo else 'is unchanged')
        print(f"Kappa {direction} from {k_lo:.3f} ({TIERS[0]}) to {k_hi:.3f} ({TIERS[-1]}).")
    if None not in (p_lo, p_hi):
        direction = 'falls' if p_hi < p_lo else ('rises' if p_hi > p_lo else 'is unchanged')
        print(f"Time MAPE {direction} from {p_lo:.1f}% ({TIERS[0]}) to {p_hi:.1f}% ({TIERS[-1]}).")
    print('\nUse whichever direction actually appears; if accuracy does NOT improve '
          'with tier, that supports the paper\'s claim that the binding constraint '
          'is the underdetermined interview, not model capability.')